In [1]:
import sqlite3
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import google.generativeai as genai
import os
from dotenv import load_dotenv
import pandas as pd

C:\Users\Dell\AppData\Local\Temp\ipykernel_11832\1664511107.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
db_path = "upload_db/Chinook.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print("Database Connected Successfully")

Database Connected Successfully


In [3]:
def load_schema(cursor):
    documents = []

    # Get all table names
    cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """)
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        # Get schema for current table
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        # Create readable schema text
        schema_text = f"Table: {table_name}\n\nColumns:\n"

        for column in columns:
            schema_text += f"{column[1]} ({column[2]})\n"

        # Convert to LangChain Document
        document = Document(
            page_content=schema_text,
            metadata={
                "table_name": table_name
            }
        )

        documents.append(document)

    return documents

In [4]:
def split_documents(documents):
    spliter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200
    )
    chunks = spliter.split_documents(documents)
    return chunks

In [5]:
embeder = SentenceTransformer("all-MiniLM-L6-v2")
def embed_chunks(chunks):
    chunks_text = [chunk.page_content for chunk in chunks]
    embeddings = embeder.encode(chunks_text)
    return embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
def store_embeddings(embeddings,chunks):
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_or_create_collection(name = "schema_collection")
    collection.add(
        ids = [str(i) for i in range(len(embeddings))],
        embeddings= embeddings.tolist(),
        documents = [chunk.page_content for chunk in chunks],
        metadatas = [chunk.metadata for chunk in chunks]
    )
    return collection

In [7]:
def ingestion_pipeline(cursor):
    documents = load_schema(cursor)
    chunks = split_documents(documents)
    embeddings = embed_chunks(chunks)
    collection = store_embeddings(embeddings,chunks)
    return collection

In [8]:
def embed_query(query):
    query_embeddings = embeder.encode(query)
    return query_embeddings

In [9]:
def retrive_schema(query_embedding,collection):
    result = collection.query(
        query_embeddings = [query_embedding.tolist()],
        n_results = 3
    )
    return result

In [10]:
def retrival_pipeline(query,collection):
    query_embedding = embed_query(query)
    retrived_schema = retrive_schema(query_embedding,collection)
    return retrived_schema

In [11]:
collection = ingestion_pipeline(cursor)

In [12]:
def generate_sql_query(query,retrived_schema):
    context = "\n\n".join(
        retrived_schema["documents"][0]
    )
    prompt = f"""
    You are an expert SQLite SQL generator.
    Your task is to generate a valid SQLite SQL query based ONLY on the provided database schema.
    Rules:
    1. Use ONLY the tables and columns present in the schema.
    2. Generate ONLY SQLite-compatible SQL.
    3. Do NOT assume tables or columns that are not in the schema.
    4. Return ONLY the SQL query.
    5. Do NOT include explanations.
    6. Do NOT use markdown or ```sql``` code fences.
    7. If the question cannot be answered using the provided schema, return exactly:
   CANNOT_GENERATE_SQL
    Database Schema:
    {context}
    User Question:
    {query}
    SQL Query:
    """
    load_dotenv()
    genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
    model = genai.GenerativeModel("gemini-2.5-flash")
    response = model.generate_content(prompt)
    return response.text.strip()

In [13]:
allowed_operation = {
    "SELECT",
    "UPDATE",
    "INSERT",
    "DELETE",
    "CREATE",
    "ALTER"
}
def validate_sql_query(sql_query):
    sql_query= sql_query.strip()
    operation=sql_query.split()[0].upper()
    if operation in allowed_operation :
        return{
            "is_valid" : True,
            "operation" : operation
        }
    else :
        return{
            "is_valid" : False,
            "operation" : operation
        }

In [14]:
def execute_sql_query(sql_query, validation, conn):

    if not validation["is_valid"]:
        return {
            "status": "Error",
            "message": "Invalid SQL Query"
        }

    try:

        if validation["operation"] == "SELECT":

            df = pd.read_sql_query(sql_query, conn)

            return {
                "status": "Success",
                "operation": validation["operation"],
                "data": df
            }

        else:

            cursor = conn.cursor()
            cursor.execute(sql_query)
            conn.commit()

            return {
                "status": "Success",
                "operation": validation["operation"],
                "rows affected": cursor.rowcount
            }

    except Exception as e:

        conn.rollback()

        return {
            "status": "Error",
            "message": str(e)
        }

In [15]:
def backend_pipeline(query,collection,conn):
    retrived_schema = retrival_pipeline(query,collection)
    sql_query = generate_sql_query(query,retrived_schema)
    validation = validate_sql_query(sql_query)
    result = execute_sql_query(sql_query,validation,conn)
    return {
    "sql_query": sql_query,
    "result": result
    }